# 📊 W8: Google Sheets 실시간 KPI 연동
**hexa-1 — Week 8** | ⏱️ 60분 | 🎯 KPI → Sheets 자동 입력

> 💡 API 없이도 CSV 저장 모드로 실습 가능

## Step 0: 환경 설정

In [ ]:
!pip install -q pandas gspread google-auth
print('✅ 패키지 설치 완료')


## Step 1: 인증 설정 (선택)

```
1. Google Cloud Console → 서비스 계정 생성
2. JSON 키 파일 다운로드
3. 아래 셀에서 키 파일 업로드
```

In [ ]:
import pandas as pd, os, json
from datetime import datetime

# Sheets 연동 설정 (API 없으면 CSV fallback)
SHEETS_CONFIG = {
    'spreadsheet_name': '✏️ [스프레드시트 이름]',
    'sheet_name': 'KPI_Daily',
    'use_api': False,  # True: 실제 Sheets 연동 / False: CSV 저장
}

# KPI 데이터 (GitHub에서 로드)
import requests, io
url = 'https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/scripts/oee_sample_data.csv'
try:
    df = pd.read_csv(url, encoding='utf-8-sig')
    print(f'✅ OEE 샘플 데이터 로드: {len(df)}행')
except:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(io.BytesIO(list(up.values())[0]), encoding='utf-8-sig')
df.head(3)


## Step 2: Sheets 연동 또는 CSV 저장

In [ ]:
def write_to_sheets(df, config):
    if not config['use_api']:
        out = f"kpi_{config['sheet_name']}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
        df.to_csv(out, index=False, encoding='utf-8-sig')
        print(f'💾 CSV 저장 (API 없음 모드): {out}')
        return out
    try:
        import gspread
        from google.colab import auth
        auth.authenticate_user()
        from google.auth import default
        creds, _ = default()
        gc = gspread.authorize(creds)
        sh = gc.open(config['spreadsheet_name'])
        ws = sh.worksheet(config['sheet_name'])
        ws.clear()
        ws.update([df.columns.tolist()] + df.values.tolist())
        print(f'✅ Google Sheets 업데이트 완료: {len(df)}행')
    except Exception as e:
        print(f'⚠️ Sheets 연동 실패: {e}')
        return write_to_sheets.__wrapped__(df, {**config, 'use_api': False})

result = write_to_sheets(df, SHEETS_CONFIG)
print('완료!')


## Step 3: 집계 및 요약 시트 작성

In [ ]:
# 주간 집계
date_col = next((c for c in df.columns if '날짜' in c), df.columns[0])
df[date_col] = pd.to_datetime(df[date_col])
df['주차'] = df[date_col].dt.isocalendar().week
numeric_cols = df.select_dtypes(include='number').columns.tolist()
weekly = df.groupby('주차')[numeric_cols].mean().round(2)
print('📅 주간 평균 KPI:')
print(weekly.to_string())
weekly.to_csv('weekly_kpi_summary.csv', encoding='utf-8-sig')
from google.colab import files
files.download('weekly_kpi_summary.csv')
print('✅ 주간 KPI 요약 다운로드 완료')


---
## 🔥 확장 과제
1. `use_api: True`로 바꾸고 실제 Sheets에 업로드
2. 자동화 트리거: Colab 스케줄러 or Google Apps Script 연동
3. `sheets_kpi_template.py` CLI로 매일 자동 실행 설정